# Seminar 3. Classification with MLP

In [ ]:
import numpy as np
import pandas as pd

import nltk

from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings("ignore")


In [ ]:
lemmatizer = WordNetLemmatizer()
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

Dataset: https://www.kaggle.com/datasets/kundanbedmutha/customer-sentiment-dataset

In [ ]:
df = pd.read_csv('/kaggle/input/customer-sentiment-dataset/Customer_Sentiment.csv')
df


,customer_id,gender,age_group,region,product_category,purchase_channel,platform,customer_rating,review_text,sentiment,response_time_hours,issue_resolved,complaint_registered
0,1,male,60+,north,automobile,online,flipkart,1,very disappointed with the quality.,negative,46,yes,yes
1,2,other,46-60,central,books,online,swiggy instamart,5,fast delivery and great packaging.,positive,5,yes,no
2,3,female,36-45,east,sports,online,facebook marketplace,1,very disappointed with the quality.,negative,38,yes,yes
3,4,female,18-25,central,groceries,online,zepto,2,product stopped working after few days.,negative,16,yes,yes
4,5,female,18-25,east,electronics,online,croma,3,neutral about the quality.,neutral,15,yes,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,24996,female,36-45,south,beauty,online,lenskart,1,very disappointed with the quality.,negative,40,yes,yes
24996,24997,other,60+,central,automobile,online,flipkart,5,"amazing experience, highly recommend!",positive,25,yes,no
24997,24998,male,18-25,south,beauty,online,ajio,4,fast delivery and great packaging.,positive,9,yes,no
24998,24999,female,26-35,central,automobile,online,snapdeal,5,great value for money.,positive,65,no,no


First, let's check our dataset

In [ ]:
df['sentiment'].value_counts()


sentiment
positive    9978
negative    9937
neutral     5085
Name: count, dtype: int64

Something else?

In [ ]:
# ideas:
# 1. words distribution over classes
# 2. text lengths distribution


Let's create a copy of a dataset for binary classification

In [ ]:
df_bin = df.loc[df['sentiment'] != 'neutral'].reset_index()
df_bin


,index,customer_id,gender,age_group,region,product_category,purchase_channel,platform,customer_rating,review_text,sentiment,response_time_hours,issue_resolved,complaint_registered
0,0,1,male,60+,north,automobile,online,flipkart,1,very disappointed with the quality.,negative,46,yes,yes
1,1,2,other,46-60,central,books,online,swiggy instamart,5,fast delivery and great packaging.,positive,5,yes,no
2,2,3,female,36-45,east,sports,online,facebook marketplace,1,very disappointed with the quality.,negative,38,yes,yes
3,3,4,female,18-25,central,groceries,online,zepto,2,product stopped working after few days.,negative,16,yes,yes
4,5,6,other,26-35,central,sports,online,facebook marketplace,5,"amazing experience, highly recommend!",positive,10,yes,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19910,24994,24995,female,26-35,north,books,online,nykaa,5,great value for money.,positive,61,no,no
19911,24995,24996,female,36-45,south,beauty,online,lenskart,1,very disappointed with the quality.,negative,40,yes,yes
19912,24996,24997,other,60+,central,automobile,online,flipkart,5,"amazing experience, highly recommend!",positive,25,yes,no
19913,24997,24998,male,18-25,south,beauty,online,ajio,4,fast delivery and great packaging.,positive,9,yes,no


## Part 1. Binary Classification

Let's encode our labels via `factorize()` method

In [ ]:
pd.factorize(df_bin['sentiment'])


(array([0, 1, 0, ..., 1, 1, 1]),
 Index(['negative', 'positive'], dtype='object'))

In [ ]:
fact, fact_keys = pd.factorize(df_bin['sentiment'])
df_bin['sentiment'] = fact
df_bin


,index,customer_id,gender,age_group,region,product_category,purchase_channel,platform,customer_rating,review_text,sentiment,response_time_hours,issue_resolved,complaint_registered
0,0,1,male,60+,north,automobile,online,flipkart,1,very disappointed with the quality.,0,46,yes,yes
1,1,2,other,46-60,central,books,online,swiggy instamart,5,fast delivery and great packaging.,1,5,yes,no
2,2,3,female,36-45,east,sports,online,facebook marketplace,1,very disappointed with the quality.,0,38,yes,yes
3,3,4,female,18-25,central,groceries,online,zepto,2,product stopped working after few days.,0,16,yes,yes
4,5,6,other,26-35,central,sports,online,facebook marketplace,5,"amazing experience, highly recommend!",1,10,yes,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19910,24994,24995,female,26-35,north,books,online,nykaa,5,great value for money.,1,61,no,no
19911,24995,24996,female,36-45,south,beauty,online,lenskart,1,very disappointed with the quality.,0,40,yes,yes
19912,24996,24997,other,60+,central,automobile,online,flipkart,5,"amazing experience, highly recommend!",1,25,yes,no
19913,24997,24998,male,18-25,south,beauty,online,ajio,4,fast delivery and great packaging.,1,9,yes,no


Now we need to preprocess our texts

In [ ]:
from tqdm.auto import tqdm
import re


In [ ]:
tqdm.pandas()

def preprocess(text):
    word_tokens = word_tokenize(text)
    filtered_text = [lemmatizer.lemmatize(w) for w in word_tokens]

    res_text = ' '.join(filtered_text)
    res_text = manual_clear(res_text)
    return res_text

def manual_clear(text):
    filtered_tokens = re.findall(r'[a-z]{2,}', text.lower())
    return ' '.join(filtered_tokens)


In [ ]:
df_bin['review_text'] = df_bin['review_text'].progress_apply(preprocess)
df_bin.head()


  0%|          | 0/19915 [00:00<?, ?it/s]

,index,customer_id,gender,age_group,region,product_category,purchase_channel,platform,customer_rating,review_text,sentiment,response_time_hours,issue_resolved,complaint_registered
0,0,1,male,60+,north,automobile,online,flipkart,1,very disappointed with the quality,0,46,yes,yes
1,1,2,other,46-60,central,books,online,swiggy instamart,5,fast delivery and great packaging,1,5,yes,no
2,2,3,female,36-45,east,sports,online,facebook marketplace,1,very disappointed with the quality,0,38,yes,yes
3,3,4,female,18-25,central,groceries,online,zepto,2,product stopped working after few day,0,16,yes,yes
4,5,6,other,26-35,central,sports,online,facebook marketplace,5,amazing experience highly recommend,1,10,yes,no


Next step: vectorization

We'll use [Doc2Vec](https://radimrehurek.com/gensim/models/doc2vec.html) for it.

Additional source: https://www.geeksforgeeks.org/nlp/doc2vec-in-nlp/?ysclid=mj1e6rlgv373510954

In [ ]:
from gensim.models.doc2vec import Doc2Vec,TaggedDocument


In [ ]:
tagged_documents = [TaggedDocument(words=doc.split(), tags=[str(i)]) for i, doc in enumerate(df_bin['review_text'])]

model = Doc2Vec(vector_size=100, min_count=2, epochs=20, dm=1, seed=42, workers=2)
model.build_vocab(tagged_documents)
model.train(tagged_documents,
            total_examples=model.corpus_count,
            epochs=model.epochs)

vectors = [model.infer_vector(doc.split()) for doc in df_bin['review_text']]


Time to split our data

In [ ]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
X = np.array(vectors)
y = df_bin['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
y_test_tensor = torch.FloatTensor(y_test)


In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


Now, let's build our neural network!

In [ ]:
import torch.nn as nn
import torch.optim as optim


In [ ]:
class SimpleModel(nn.Module):
    def __init__(self, input_size):
        super(SimpleModel, self).__init__()
        self.linear = nn.Linear(input_size, 1)
    
    def forward(self, x):
        return self.linear(x)

# Initialize model
simple_model = SimpleModel(X.shape[1])
print("Model architecture:")
print(simple_model)
print(f"\nNumber of parameters: {sum(p.numel() for p in simple_model.parameters())}")


Model architecture:
SimpleModel(
  (linear): Linear(in_features=100, out_features=1, bias=True)
)

Number of parameters: 101


In [ ]:
criterion = nn.BCEWithLogitsLoss()  # Combines Sigmoid + BCELoss
optimizer = optim.Adam(simple_model.parameters(), lr=0.01)


Okay, let's train our model 

In [ ]:
num_epochs = 10

print("Training...")
for epoch in range(num_epochs):
    simple_model.train()
    running_loss = 0.0
    
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
    
    for batch_idx, (inputs, labels) in enumerate(train_pbar):
        optimizer.zero_grad()
        
        # Forward pass (linear transformation only)
        outputs = simple_model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        train_pbar.set_postfix({'loss': running_loss/(batch_idx+1)})
    
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}')


Training...


Epoch 1/10:   0%|          | 0/249 [00:00<?, ?it/s]

Epoch 1/10, Loss: 0.4904


Epoch 2/10:   0%|          | 0/249 [00:00<?, ?it/s]

Epoch 2/10, Loss: 0.3421


...

Epoch 8/10, Loss: 0.2434


Epoch 9/10:   0%|          | 0/249 [00:00<?, ?it/s]

Epoch 9/10, Loss: 0.2367


Epoch 10/10:   0%|          | 0/249 [00:00<?, ?it/s]

Epoch 10/10, Loss: 0.2304


Time to evaluate!

In [ ]:
from sklearn.metrics import classification_report


In [ ]:
simple_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    test_pbar = tqdm(test_loader, desc='Testing')
    for inputs, labels in test_pbar:
        outputs = simple_model(inputs)
        # Apply sigmoid to get probabilities
        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).int()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Flatten lists
all_preds = [p[0] for p in all_preds]

print("\nResults:")
print("=" * 50)
print(classification_report(all_labels, all_preds, 
                           target_names=['negative', 'positive']))


Testing:   0%|          | 0/63 [00:00<?, ?it/s]


Results:
              precision    recall  f1-score   support

    negative       0.89      0.91      0.90      1987
    positive       0.91      0.89      0.90      1996

    accuracy                           0.90      3983
   macro avg       0.90      0.90      0.90      3983
weighted avg       0.90      0.90      0.90      3983



Cool! But this was not a neural network. `simple_model` is a logistic regression

Ok, let's make a real neural network this time

**NB!** Here we explicitly use sigmoid in our neural network, so instead of binary cross-entropy with logits we use simple binary cross-entropy

In [ ]:
class BinaryMLP(nn.Module):
    def __init__(self, input_size):
        super(BinaryMLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.layers(x)

# Initialize model
model = BinaryMLP(X.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    # Progress bar for training
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
    
    for batch_idx, (inputs, labels) in enumerate(train_pbar):
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Update progress bar
        train_pbar.set_postfix({'loss': running_loss/(batch_idx+1)})
    
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}')


Epoch 1/10:   0%|          | 0/249 [00:00<?, ?it/s]

Epoch 1/10, Loss: 0.4147


Epoch 2/10:   0%|          | 0/249 [00:00<?, ?it/s]

Epoch 2/10, Loss: 0.2224


Epoch 3/10:   0%|          | 0/249 [00:00<?, ?it/s]

...

Epoch 8/10, Loss: 0.0117


Epoch 9/10:   0%|          | 0/249 [00:00<?, ?it/s]

Epoch 9/10, Loss: 0.0076


Epoch 10/10:   0%|          | 0/249 [00:00<?, ?it/s]

Epoch 10/10, Loss: 0.0056


In [ ]:
# Evaluation
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    test_pbar = tqdm(test_loader, desc='Testing')
    for inputs, labels in test_pbar:
        outputs = model(inputs)
        preds = (outputs >= 0.5).int()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Flatten lists
all_preds = [p[0] for p in all_preds]

print(classification_report(all_labels, all_preds, 
                           target_names=['negative', 'positive']))


Testing:   0%|          | 0/63 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      1987
    positive       1.00      1.00      1.00      1996

    accuracy                           1.00      3983
   macro avg       1.00      1.00      1.00      3983
weighted avg       1.00      1.00      1.00      3983



## Part 2. Multiclass Classification

In [ ]:
# Encode labels using mapping
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
df['sentiment'] = df['sentiment'].map(label_mapping)
df.tail()


,customer_id,gender,age_group,region,product_category,purchase_channel,platform,customer_rating,review_text,sentiment,response_time_hours,issue_resolved,complaint_registered,sentiment_encoded
24995,24996,female,36-45,south,beauty,online,lenskart,1,very disappointed with the quality.,0,40,yes,yes,0
24996,24997,other,60+,central,automobile,online,flipkart,5,"amazing experience, highly recommend!",2,25,yes,no,2
24997,24998,male,18-25,south,beauty,online,ajio,4,fast delivery and great packaging.,2,9,yes,no,2
24998,24999,female,26-35,central,automobile,online,snapdeal,5,great value for money.,2,65,no,no,2
24999,25000,male,46-60,central,travel,online,lenskart,3,"product is okay, nothing special.",1,67,no,no,1


In [ ]:
# Preprocess texts
df['review_text'] = df['review_text'].progress_apply(preprocess)
df.head()


  0%|          | 0/25000 [00:00<?, ?it/s]

,customer_id,gender,age_group,region,product_category,purchase_channel,platform,customer_rating,review_text,sentiment,response_time_hours,issue_resolved,complaint_registered,sentiment_encoded
0,1,male,60+,north,automobile,online,flipkart,1,very disappointed with the quality,0,46,yes,yes,0
1,2,other,46-60,central,books,online,swiggy instamart,5,fast delivery and great packaging,2,5,yes,no,2
2,3,female,36-45,east,sports,online,facebook marketplace,1,very disappointed with the quality,0,38,yes,yes,0
3,4,female,18-25,central,groceries,online,zepto,2,product stopped working after few day,0,16,yes,yes,0
4,5,female,18-25,east,electronics,online,croma,3,neutral about the quality,1,15,yes,no,1


In [ ]:
# Vectorization with Doc2Vec
# Initialize and train model in one line
multiclass_model = Doc2Vec(
    documents=[TaggedDocument(words=doc.split(), tags=[str(i)]) 
               for i, doc in enumerate(df['review_text'])],
    vector_size=100,
    min_count=2,
    epochs=20,
    dm=1,
    seed=42,
    workers=2
)

multiclass_vectors = [multiclass_model.infer_vector(doc.split()) 
                     for doc in df['review_text']]


In [ ]:
# Prepare data for training
X_multi = np.array(multiclass_vectors)
y_multi = df['sentiment_encoded'].values

# Split into train and test
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

# Convert to PyTorch tensors
X_train_m_tensor = torch.FloatTensor(X_train_m)
X_test_m_tensor = torch.FloatTensor(X_test_m)
y_train_m_tensor = torch.LongTensor(y_train_m)
y_test_m_tensor = torch.LongTensor(y_test_m)

# Create DataLoader
train_dataset_m = TensorDataset(X_train_m_tensor, y_train_m_tensor)
test_dataset_m = TensorDataset(X_test_m_tensor, y_test_m_tensor)

train_loader_m = DataLoader(train_dataset_m, batch_size=32, shuffle=True)
test_loader_m = DataLoader(test_dataset_m, batch_size=32, shuffle=False)


**NB!** Here we use `CrossEntropyLoss` as we are working with $>2$ classes.

**NB!** We don't explicitly use softmax here as `CrossEntropyLoss` works with raw logits.

In [ ]:
# Define MLP model for multiclass classification
class MulticlassMLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super(MulticlassMLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        return self.layers(x)

# Initialize model
num_classes = len(label_mapping)
model_multi = MulticlassMLP(X_multi.shape[1], num_classes)
criterion_multi = nn.CrossEntropyLoss()
optimizer_multi = optim.Adam(model_multi.parameters(), lr=0.001)


In [ ]:
# Training loop for multiclass
num_epochs_multi = 10

for epoch in range(num_epochs_multi):
    model_multi.train()
    running_loss = 0.0
    
    train_pbar = tqdm(train_loader_m, desc=f'Epoch {epoch+1}/{num_epochs_multi}')
    
    for batch_idx, (inputs, labels) in enumerate(train_pbar):
        optimizer_multi.zero_grad()
        
        outputs = model_multi(inputs)
        loss = criterion_multi(outputs, labels)
        
        loss.backward()
        optimizer_multi.step()
        
        running_loss += loss.item()
        train_pbar.set_postfix({'loss': running_loss/(batch_idx+1)})
    
    print(f'Epoch {epoch+1}/{num_epochs_multi}, Loss: {running_loss/len(train_loader_m):.4f}')


Epoch 1/10:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 1/10, Loss: 0.4427


Epoch 2/10:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 2/10, Loss: 0.1806


Epoch 3/10:   0%|          | 0/625 [00:00<?, ?it/s]

...

Epoch 8/10, Loss: 0.0279


Epoch 9/10:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 9/10, Loss: 0.0228


Epoch 10/10:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 10/10, Loss: 0.0211


In [ ]:
# Evaluation for multiclass
model_multi.eval()
all_preds_m = []
all_labels_m = []

with torch.no_grad():
    test_pbar = tqdm(test_loader_m, desc='Testing')
    for inputs, labels in test_pbar:
        outputs = model_multi(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds_m.extend(preds.cpu().numpy())
        all_labels_m.extend(labels.cpu().numpy())

print(classification_report(all_labels_m, all_preds_m, 
                           target_names=list(label_mapping.keys())))


Testing:   0%|          | 0/157 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    negative       0.99      0.99      0.99      1987
     neutral       0.98      0.99      0.98      1017
    positive       1.00      0.99      1.00      1996

    accuracy                           0.99      5000
   macro avg       0.99      0.99      0.99      5000
weighted avg       0.99      0.99      0.99      5000



Example: inferencing trained models

In [ ]:
# Predictions
def get_prediction(text, model_d2v, model_mlp, task='multi'):
    # Preprocess
    processed = preprocess(text)
    # Vectorize
    vector = model_d2v.infer_vector(processed.split())
    # Predict
    if task == 'multi':
        with torch.no_grad():
            tensor = torch.FloatTensor(vector).unsqueeze(0)
            output = model_mlp(tensor)
            probs = nn.functional.softmax(output)
            print(f"Output: {output}")
            print(f"Probabilities: {probs}")
            predicted = torch.argmax(output)
            print(f"Predicted: {predicted}")
    
        # Reverse mapping
        reverse_mapping = {v: k for k, v in label_mapping.items()}
        return reverse_mapping[predicted.item()]

    with torch.no_grad():
        tensor = torch.FloatTensor(vector).unsqueeze(0)
        output = model(tensor)
        print(f"Output: {output}")
        pred = (output >= 0.5).int()
        print(f"Predicted: {pred}")
    return list(fact_keys)[pred]

# Test with a sample review
sample_review = "The product was amazing, I really loved it!"
prediction = get_prediction(sample_review, multiclass_model, model_multi, task='multi')
print(f"Review: {sample_review}")
print(f"Predicted sentiment: {prediction}")


Output: tensor([[1.0000]])
Predicted: tensor([[1]], dtype=torch.int32)
Review: The product was amazing, I really loved it!
Predicted sentiment: positive
